# DeepVoid: segmenting the cosmic web with a 3D U-Net

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/skumagai3/deepvoid-demo/blob/main/notebooks/deepvoid_demo.ipynb)

This notebook runs trained DeepVoid models end to end, in a few minutes on a CPU (under a minute on a GPU):

1. load a 128³ cube from the IllustrisTNG300 dark-matter simulation (205 h⁻¹ Mpc on a side),
2. classify every one of its 2.1 million voxels as void, wall, filament or halo,
3. compare the prediction with the ground truth on validation subcubes the model never trained on,
4. repeat with only the subhalos (the sparse objects that host galaxies) as input, which is closer to what a galaxy survey sees.

The labels come from the T-web: at each point, count how many directions the local tidal field is
compressing matter (0 = void, 1 = wall, 2 = filament, 3 = halo). The network only sees the density
field and has to learn this from it.

*Simulation data: [IllustrisTNG](https://www.tng-project.org/) (TNG300-3-Dark), produced and publicly
released by the IllustrisTNG collaboration. See the README for the papers to cite.*

In [ ]:
# Get the code, data and weights. In Colab this clones the repo (about 20 MB);
# locally, run the notebook from inside the repo.
import os, subprocess
if not os.path.isdir("dvdemo"):
    if os.path.isdir("../dvdemo"):
        os.chdir("..")
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1",
                        "https://github.com/skumagai3/deepvoid-demo.git"], check=True)
        os.chdir("deepvoid-demo")
print("working in", os.getcwd())

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from dvdemo import viz
from dvdemo.data import Normalizer, split_masks
from dvdemo.inference import keras_predict_fn, predict_volume
from dvdemo.io import CLASS_NAMES, load_sample
from dvdemo.metrics import evaluate, format_report
from dvdemo.train import load_run

print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU") or "none (CPU is fine)")

## 1. The data

The sample is 4.5 MB: two input fields and the label cube, all on the same 128³ grid (1.6 h⁻¹ Mpc voxels).

* Dark matter: the full matter density, sampled by particles 0.33 h⁻¹ Mpc apart. Lots of detail, but not something we can observe.
* Subhalos at λ = 7 h⁻¹ Mpc: only the subhalos, thinned to a mean separation of 7 h⁻¹ Mpc. That's about
  10,000 times fewer tracers than the dark matter, and 93% of voxels are empty, which is much closer to what a galaxy survey sees.
* Labels: T-web classes computed from the full dark matter, used for both inputs. With the sparse input, the
  network has to infer structure it can't see directly.

In [ ]:
sample = load_sample("data/tng300_128.npz")
meta, split, mask = sample["meta"], sample["meta"]["split"], sample["mask"]
print(meta["simulation"])
print(f"box {meta['box_mpc_h']:.0f} h⁻¹ Mpc, grid {meta['grid'][0]}^3, voxel {meta['voxel_mpc_h']:.2f} h⁻¹ Mpc")
for k, f in meta["fields"].items():
    print(f"  {k:11s} {f['description']}  ({100 * f['empty_voxel_fraction']:.0f}% empty voxels)")
print("class fractions:", {c: round(v, 3) for c, v in zip(CLASS_NAMES, meta["labels"]["fractions"])})

TRAIN, VAL = split_masks(mask.shape, split)   # VAL: the voxels used for every score below
x_mid = 48                                    # this x-slice passes through 6 of the 13 validation subcubes

fig = viz.alignment_figure(sample, 64)   # middle slice through each axis
plt.show()

Voids fill most of the volume (62%) while halos are only 0.5% of voxels, so plain accuracy is misleading:
predicting "void" everywhere already scores 62%. I report void F1, since voids are the class I care about
most, and the multiclass Matthews correlation coefficient (MCC), which accounts for the class imbalance.

### How the box is split

The box is cut into 32³ subcubes (4 × 4 × 4 = 64) and the subcubes are
split at random into training (51) and validation (13). Only the training subcubes are augmented, with four
copies each (as-is and rotated 90° about each axis); validation subcubes are never rotated or trained on.
Validation loss drives learning-rate reduction and early stopping, and every number in this notebook is
computed on the validation subcubes (13 × 32³ ≈ 426,000 voxels, 20% of the box).

In [ ]:
fig, ax = plt.subplots(figsize=(5.4, 5.6))
viz.show_split(ax, mask, split, index=x_mid); plt.show()

## 2. Predict the full box with the dark-matter model

The model is a 3D U-Net (3 resolution levels, 16 → 128 filters, about 1.4 M parameters) trained on the
augmented 32³ training subcubes. The input scaling (min-max by default; the training notebook explains the
alternatives) is fitted on the training voxels and saved with the model.

At prediction time the periodic box is covered by 512 overlapping 32³ windows; only the central 16³ of
each is kept, so every voxel gets 8 voxels of context on all sides and there are no seams.

In [ ]:
model, norm, record = load_run("weights/dm_model")
print(f"{record['params']:,} parameters; input scaling: {norm.method}; "
      f"best epoch {record['train']['best_epoch']} of {record['train']['epochs_run']}")

t = time.time()
probs_dense = predict_volume(keras_predict_fn(model), norm.transform(sample["dm"]), verbose=True)
pred_dense = probs_dense.argmax(-1).astype(np.uint8)
print(f"full 128^3 box predicted in {time.time() - t:.1f} s")

In [ ]:
m_dense = evaluate(mask, pred_dense, list(CLASS_NAMES), where=VAL)
print(format_report(m_dense, CLASS_NAMES))
viz.plot_confusion(m_dense["confusion_matrix"], title="Validation subcubes, dark-matter input"); plt.show()

In [ ]:
fig = viz.compare_slice(sample["dm"], mask, pred_dense, x_mid, field_title="Dark matter, λ = 0.33 h⁻¹ Mpc",
                        split=split)
plt.show()

The dashed squares are the validation subcubes on this slice. Compare where the errors fall with the
class boundaries in the ground truth. The maps below show the model's probability of "void" and its
overall uncertainty (entropy of the predicted class probabilities) on the same slice.

In [ ]:
p_void = probs_dense[x_mid, :, :, 0]
entropy = -(probs_dense[x_mid] * np.log(np.clip(probs_dense[x_mid], 1e-7, 1))).sum(-1)
ext = (0, viz.BOX, 0, viz.BOX)
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
im = axes[0].imshow(p_void.T, origin="lower", cmap="Blues_r", vmin=0, vmax=1, extent=ext)
axes[0].set_title("P(void)"); plt.colorbar(im, ax=axes[0], fraction=0.046)
im = axes[1].imshow(entropy.T, origin="lower", cmap="viridis", extent=ext)
axes[1].set_title("Prediction entropy (uncertainty)"); plt.colorbar(im, ax=axes[1], fraction=0.046)
for a in axes: viz.label_axes(a, axis=0)
plt.tight_layout(); plt.show()

Browse the box slice by slice. Dashed squares mark the validation subcubes on each slice; everything outside them was used for training.

In [ ]:
from ipywidgets import IntSlider, interact

@interact(x=IntSlider(value=x_mid, min=0, max=mask.shape[0] - 1, description="slice x"))
def browse(x):
    viz.compare_slice(sample["dm"], mask, pred_dense, x,
                      field_title="Dark matter, λ = 0.33 h⁻¹ Mpc", split=split)
    plt.show()

## 3. Sparse subhalos (λ = 7 h⁻¹ Mpc)

Real surveys see galaxies, not dark matter. Here the input is only the subhalo field, but the target is
still the T-web of the full matter distribution. I compare three ways to handle it, alongside the
dark matter model from section 2:


| Model | Trained on | Applied to | What it tests |
|---|---|---|---|
| `dm_model` | dark matter (λ = 0.33 h⁻¹ Mpc) | dark matter | the basic task, with the full matter field as input |
| `subhalo_no_retraining` | dark matter | subhalos (λ = 7 h⁻¹ Mpc) | does a dark-matter model work on sparse tracers as-is? (input rescaled, weights unchanged) |
| `subhalo_from_scratch` | subhalos | subhalos | training directly on the sparse field, starting from random weights |
| `subhalo_curriculum` | dark matter, then subhalos | subhalos | one-step curriculum learning: start from `dm_model`, freeze its first encoder block, fine-tune on subhalos |

All four are scored against the same T-web labels (computed from the full dark matter) on the same
validation subcubes.

All models use the same split, training settings and early stopping.

In [ ]:
preds, scores = {}, {}
for name in ("subhalo_curriculum", "subhalo_from_scratch"):
    mdl, nrm, rec = load_run(f"weights/{name}")
    p = predict_volume(keras_predict_fn(mdl), nrm.transform(sample["subhalo_L7"]))
    preds[name] = p.argmax(-1).astype(np.uint8)

zs = Normalizer(norm.method).fit(sample["subhalo_L7"][TRAIN])
preds["subhalo_no_retraining"] = predict_volume(keras_predict_fn(model), zs.transform(sample["subhalo_L7"])).argmax(-1).astype(np.uint8)

labels = {"dm_model": "dm_model: dark matter → dark matter",
          "subhalo_no_retraining": "subhalo_no_retraining: dm_model applied to subhalos",
          "subhalo_from_scratch": "subhalo_from_scratch: trained on subhalos only",
          "subhalo_curriculum": "subhalo_curriculum: dm_model fine-tuned on subhalos"}
scores["dm_model"] = m_dense
for k in ("subhalo_no_retraining", "subhalo_from_scratch", "subhalo_curriculum"):
    scores[k] = evaluate(mask, preds[k], list(CLASS_NAMES), where=VAL)

cols = ["void_f1", "mcc", "balanced_accuracy", "wall_f1", "filament_f1", "halo_f1"]
table = pd.DataFrame({labels[k]: {c: scores[k][c] for c in cols} for k in labels}).T
table.style.format("{:.3f}").background_gradient(cmap="Blues", vmin=0, vmax=1)

In [ ]:
fig, ax = plt.subplots(figsize=(7.4, 3.8))
viz.plot_model_comparison(scores, labels, ax=ax); plt.tight_layout(); plt.show()
fig = viz.compare_slice(sample["subhalo_L7"], mask, preds["subhalo_curriculum"], x_mid,
                        field_title="Subhalos, λ = 7 h⁻¹ Mpc", sparse=True, split=split)
plt.show()

## Caveats

* This is a scaled-down version of the method. The full-scale results (Kumagai et al., ApJ 2026) are
  void F1 = 0.96 for dark matter at λ = 0.33 h⁻¹ Mpc and void F1 = 0.89 for subhalos at λ = 10 h⁻¹ Mpc.
  To fit in a repo and train in minutes, this demo uses a 4× coarser grid (1.6 instead of 0.4 h⁻¹ Mpc
  voxels), 64× less data, a small U-Net and very little tuning, so its numbers aren't directly
  comparable. The README has more detail.
* The curriculum is simplified. The demo goes from dark matter (λ = 0.33) straight to subhalos at
  λ = 7 h⁻¹ Mpc in one step. The paper's best scheme is two-step (λ = 0.33 → 3 → 10 h⁻¹ Mpc) on much
  sparser tracers in voxel terms, which is where pretraining should help most. Even there the gain over
  direct training is modest (MCC 0.60 vs. 0.56, same void F1 of 0.89), so a near-tie here isn't surprising.
* It's one simulation. Good scores here show the model generalizes to unseen regions of the same
  simulation, not to other cosmologies or to real surveys (which add redshift-space distortions, survey masks
  and selection effects).
* This is a segmentation, not a void catalog. Turning voxel labels into individual voids (centers, radii)
  needs a separate, well-defined post-processing step.

Links: [paper (arXiv:2504.21134)](https://arxiv.org/abs/2504.21134) ·
[research code](https://github.com/skumagai3/deepvoid_misc) ·
[DV_NEW rewrite](https://github.com/skumagai3/DV_NEW) ·
[how the demo models were trained](train_demo_models.ipynb)